# RAG System with LangChain and LCEL

This notebook implements a RAG system using:
- **LangChain** for document loading, splitting, and chains
- **ChatOpenAI** for generation (LCEL compatible)
- **OpenAIEmbeddings** for embeddings
- **ChromaDB** for vector storage

## Cell 1: Install Requirements

In [ ]:
!pip install -r requirements.txt -q

## Cell 2: Import Libraries

In [1]:
import os
from pathlib import Path
from typing import List

# LangChain imports
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    Docx2txtLoader,
    UnstructuredFileLoader
)
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

print("All libraries imported successfully!")

/Users/udaybijjala/miniconda3/envs/myenv_rag/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


All libraries imported successfully!


## Cell 3: Set OpenAI API Key and Initialize Models

In [2]:
# Set your OpenAI API key here
OPENAI_API_KEY = "API_KEY"  # Replace with your actual API key
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Initialize LangChain ChatOpenAI (GPT-3.5)
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.7,
    max_tokens=1000
)

# Initialize LangChain OpenAI Embeddings (Ada)
embeddings = OpenAIEmbeddings(
    model="text-embedding-ada-002"
)

print(f"LLM Model: {llm.model_name}")
print(f"Embeddings Model: {embeddings.model}")
print("Models initialized successfully!")

LLM Model: gpt-3.5-turbo
Embeddings Model: text-embedding-ada-002
Models initialized successfully!


## Cell 4: Document Loading Functions

In [3]:
DOCUMENTS_FOLDER = "./documents"

def load_document(file_path: str):
    """Load a document based on its file extension."""
    file_extension = Path(file_path).suffix.lower()
    
    try:
        if file_extension == ".pdf":
            loader = PyPDFLoader(file_path)
        elif file_extension == ".txt":
            loader = TextLoader(file_path, encoding="utf-8")
        elif file_extension in [".docx", ".doc"]:
            loader = Docx2txtLoader(file_path)
        else:
            loader = UnstructuredFileLoader(file_path)
        
        return loader.load()
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return []


def load_all_documents(folder_path: str):
    """Load all documents from the specified folder."""
    documents = []
    folder = Path(folder_path)
    file_stats = {}
    
    if not folder.exists():
        print(f"Folder '{folder_path}' does not exist!")
        return documents
    
    supported_extensions = [".pdf", ".txt", ".docx", ".doc", ".md"]
    file_count = 0
    
    for file_path in folder.iterdir():
        if file_path.is_file() and file_path.suffix.lower() in supported_extensions:
            file_count += 1
            print(f"Loading: {file_path.name}")
            docs = load_document(str(file_path))
            
            # Add document name to metadata
            for doc in docs:
                doc.metadata["document_name"] = file_path.name
            
            documents.extend(docs)
            file_stats[file_path.name] = len(docs)
    
    print(f"\n{'='*50}")
    print(f"Files loaded: {file_count}")
    print(f"Total pages/documents: {len(documents)}")
    print(f"{'='*50}")
    print("\nBreakdown:")
    for filename, page_count in file_stats.items():
        print(f"  {filename}: {page_count} pages")
    print(f"{'='*50}\n")
    
    return documents

print("Document loading functions defined.")

Document loading functions defined.


## Cell 5: Text Splitting Function

In [4]:
def split_documents(documents, chunk_size: int = 1000, chunk_overlap: int = 200):
    """Split documents into smaller chunks."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    chunks = text_splitter.split_documents(documents)
    print(f"Documents split into {len(chunks)} chunks")
    return chunks

print("Text splitting function defined.")

Text splitting function defined.


## Cell 6: Initialize ChromaDB with LangChain

In [5]:
CHROMA_DB_PATH = "./chroma_db_langchain"
COLLECTION_NAME = "document_chunks_langchain"

# Global variable for vectorstore
vectorstore = None

def create_vectorstore(chunks):
    """Create ChromaDB vectorstore from document chunks using LangChain."""
    global vectorstore
    
    print("Creating ChromaDB vectorstore with LangChain...")
    print(f"Generating embeddings for {len(chunks)} chunks...")
    
    # Create vectorstore from documents (embeddings generated automatically)
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=CHROMA_DB_PATH
    )
    
    print(f"Vectorstore created with {vectorstore._collection.count()} chunks!")
    print(f"Persisted to: {CHROMA_DB_PATH}")
    return vectorstore


def load_existing_vectorstore():
    """Load existing ChromaDB vectorstore."""
    global vectorstore
    
    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=CHROMA_DB_PATH
    )
    
    count = vectorstore._collection.count()
    print(f"Loaded existing vectorstore with {count} chunks")
    return vectorstore

print("ChromaDB functions defined.")

ChromaDB functions defined.


## Cell 7: Load, Split, and Store Documents

In [6]:
# Load all documents from the documents folder
print("=" * 50)
print("STEP 1: Loading Documents")
print("=" * 50)
documents = load_all_documents(DOCUMENTS_FOLDER)

if documents:
    print("=" * 50)
    print("STEP 2: Splitting Documents into Chunks")
    print("=" * 50)
    chunks = split_documents(documents, chunk_size=1000, chunk_overlap=200)
    
    print("\n" + "=" * 50)
    print("STEP 3: Creating Vectorstore with Embeddings")
    print("=" * 50)
    vectorstore = create_vectorstore(chunks)
    
    print("\n" + "=" * 50)
    print("Document processing complete!")
    print("=" * 50)
else:
    print("\nNo documents found. Add documents to:", os.path.abspath(DOCUMENTS_FOLDER))

STEP 1: Loading Documents
Loading: RFP SM-PRC0001034 Addendum 1.pdf
Loading: Scope responses for ID Verification.pdf
Loading: Alamo College RFP V1.2.pdf


/Users/udaybijjala/miniconda3/envs/myenv_rag/lib/python3.10/site-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


Loading: RFP_Som-AJKEditsv1.pdf

Files loaded: 4
Total pages/documents: 41

Breakdown:
  RFP SM-PRC0001034 Addendum 1.pdf: 3 pages
  Scope responses for ID Verification.pdf: 2 pages
  Alamo College RFP V1.2.pdf: 20 pages
  RFP_Som-AJKEditsv1.pdf: 16 pages

STEP 2: Splitting Documents into Chunks
Documents split into 61 chunks

STEP 3: Creating Vectorstore with Embeddings
Creating ChromaDB vectorstore with LangChain...
Generating embeddings for 61 chunks...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vectorstore created with 61 chunks!
Persisted to: ./chroma_db_langchain

Document processing complete!


## Cell 8: Create Retriever

In [8]:
# Create retriever from vectorstore (top 5 results)
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

print("Retriever created (top 5 similar chunks)")

Retriever created (top 5 similar chunks)


## Cell 9: Define RAG Prompt Template

In [9]:
# RAG prompt template
template = """You are a helpful assistant that answers questions based on the provided context.
Always base your answers on the given context. If the context doesn't contain enough information
to answer the question, say so. When possible, cite the source document for your information.

Context:
{context}

Question: {question}

Please provide a comprehensive answer based on the context above. Cite the source documents when relevant."""

prompt = ChatPromptTemplate.from_template(template)

print("RAG prompt template defined.")

RAG prompt template defined.


## Cell 10: Build LCEL RAG Chain

In [10]:
def format_docs(docs):
    """Format retrieved documents with source information."""
    formatted = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("document_name", "unknown")
        formatted.append(f"[Source {i}: {source}]\n{doc.page_content}")
    return "\n\n".join(formatted)


# Build the LCEL RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("LCEL RAG chain created!")
print("\nChain structure:")
print("  Query -> Retriever -> Format Docs -> Prompt -> LLM -> Output")

LCEL RAG chain created!

Chain structure:
  Query -> Retriever -> Format Docs -> Prompt -> LLM -> Output


## Cell 11: RAG Query Function

In [11]:
def rag_query(query: str, show_chunks: bool = True) -> str:
    """Execute RAG query using LCEL chain."""
    
    print(f"\nQuery: {query}")
    print("-" * 60)
    
    # Optionally show retrieved chunks
    if show_chunks:
        print("\nStep 1: Retrieving relevant chunks...")
        docs = retriever.invoke(query)
        
        print("\n" + "=" * 60)
        print("RETRIEVED CHUNKS (Top 5)")
        print("=" * 60)
        
        for i, doc in enumerate(docs, 1):
            print(f"\n--- Chunk {i} ---")
            print(f"Document: {doc.metadata.get('document_name', 'unknown')}")
            content = doc.page_content
            print(f"Content Preview: {content[:300]}..." if len(content) > 300 else f"Content: {content}")
        
        print("\n" + "=" * 60)
    
    # Generate answer using LCEL chain
    print("\nStep 2: Generating answer using LCEL chain...")
    answer = rag_chain.invoke(query)
    
    print("\n" + "=" * 60)
    print("GENERATED ANSWER")
    print("=" * 60)
    print(answer)
    print("=" * 60)
    
    return answer

print("RAG query function defined.")

RAG query function defined.


## Cell 12: Test the RAG System

In [12]:
# Example query - replace with your own question
user_query = "What is the main topic of the documents?"

# Run the RAG query
answer = rag_query(user_query, show_chunks=True)


Query: What is the main topic of the documents?
------------------------------------------------------------

Step 1: Retrieving relevant chunks...


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



RETRIEVED CHUNKS (Top 5)

--- Chunk 1 ---
Document: Alamo College RFP V1.2.pdf
Content Preview: d) Provide  efforts  the firm makes  to keep its professionals  informed  of 
developments relevant to the industry.  
3) Extent  to Which  the Goods  or Services  Meet  the District’s  Needs:  
a) A brief discussion of your firm’s background and experience in providing the  
requested goods and ser...

--- Chunk 2 ---
Document: Alamo College RFP V1.2.pdf
Content Preview: ExamRoom.AI  Page  | 2  
 
 
CONTENTS  
 
 
MINIMUM  QUALIFICATIONS  ................................ ................................ ................................ ................................ ........  3 
SECTION 1  SCOPE  OF WORK ................................ ..............................

--- Chunk 3 ---
Document: RFP SM-PRC0001034 Addendum 1.pdf
Content Preview: navigation or downloads required, if possible. 
 
12.Q. Does WGU recommend any specific APIs to check and validate student government issued IDs?  

## Cell 13: Streaming Response (LCEL Feature)

In [14]:
def rag_query_stream(query: str):
    """Execute RAG query with streaming response using LCEL."""
    
    print(f"\nQuery: {query}")
    print("-" * 60)
    print("\nStreaming response:\n")
    
    # Stream the response
    full_response = ""
    for chunk in rag_chain.stream(query):
        print(chunk, end="", flush=True)
        full_response += chunk
    
    print("\n" + "=" * 60)
    return full_response

# Test streaming
answer = rag_query_stream("What is the main topic?")
print("Streaming function defined. Uncomment above line to test.")


Query: What is the main topic?
------------------------------------------------------------

Streaming response:

Based on the context provided from the Alamo College RFP V1.2.pdf document, the main topic is the development and implementation of a solution by ExamRoom.AI to improve processing times, ensure compliance with state-specific requirements, and enhance the user experience for staff and applicants. The solution involves the use of core technologies such as Open CV, NLP & LLM, frameworks like LangChain, LamaIndex, Hugging Face agents, and programming language Python. The architecture design includes cloud-based AI and microservices.

Additionally, the hosting requirements and security measures for the solution are outlined, with a focus on ongoing maintenance. The resources and processes required to maintain the final delivered solution are detailed, including security maintenance, scalability, performance, availability, uptime, issue resolution, and maintenance processes. The

## Cell 14: Chain with Sources (LCEL Feature)

In [ ]:
# RAG chain that returns both answer and source documents
rag_chain_with_sources = RunnableParallel(
    {
        "answer": rag_chain,
        "sources": retriever
    }
)

def rag_query_with_sources(query: str):
    """Execute RAG query and return answer with sources."""
    
    print(f"\nQuery: {query}")
    print("-" * 60)
    
    result = rag_chain_with_sources.invoke(query)
    
    print("\n" + "=" * 60)
    print("GENERATED ANSWER")
    print("=" * 60)
    print(result["answer"])
    
    print("\n" + "=" * 60)
    print("SOURCES")
    print("=" * 60)
    unique_sources = set()
    for doc in result["sources"]:
        unique_sources.add(doc.metadata.get("document_name", "unknown"))
    for source in unique_sources:
        print(f"  - {source}")
    print("=" * 60)
    
    return result

print("RAG chain with sources defined.")

## Cell 15: Interactive Query Mode

In [ ]:
def interactive_query():
    """Run queries interactively in a loop."""
    print("=" * 60)
    print("INTERACTIVE RAG QUERY MODE (LangChain LCEL)")
    print("Type 'quit' or 'exit' to stop")
    print("=" * 60)
    
    while True:
        query = input("\nEnter your question: ").strip()
        
        if query.lower() in ["quit", "exit", "q"]:
            print("\nGoodbye!")
            break
        
        if not query:
            print("Please enter a valid question.")
            continue
        
        rag_query(query, show_chunks=False)

# Uncomment the line below to start interactive mode
# interactive_query()

## Cell 16: Utility Functions

In [ ]:
def get_vectorstore_stats():
    """Display statistics about the vectorstore."""
    if vectorstore is None:
        print("Vectorstore not initialized.")
        return 0
    
    count = vectorstore._collection.count()
    print(f"Total chunks in vectorstore: {count}")
    
    # Get sample documents
    if count > 0:
        sample = vectorstore.similarity_search("test", k=min(count, 5))
        unique_docs = set()
        for doc in sample:
            unique_docs.add(doc.metadata.get("document_name", "unknown"))
        print(f"Sample documents: {unique_docs}")
    
    return count


def clear_vectorstore():
    """Clear all data from the vectorstore."""
    import shutil
    if os.path.exists(CHROMA_DB_PATH):
        shutil.rmtree(CHROMA_DB_PATH)
        print(f"Cleared vectorstore at {CHROMA_DB_PATH}")
    else:
        print("Vectorstore directory does not exist.")


# Display current stats
get_vectorstore_stats()